<a href="https://colab.research.google.com/github/tipismaca/Trabajo-Bebidas-del-Maipo-SA/blob/main/Codigo/Entrega2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Caso de Estudio: Bebidas del Maipo S.A. — Entrega 2
**Autoras:** Maytte Almuna, Macarena Campos, Darlyn Jerez y Valentina Roa


## 1. Configuración inicial
Clonación del repositorio y librerías

In [1]:
# Clona el repositorio para tener acceso a los datos
!rm -rf Trabajo-Bebidas-del-Maipo-SA
!git clone https://github.com/tipismaca/Trabajo-Bebidas-del-Maipo-SA.git

Cloning into 'Trabajo-Bebidas-del-Maipo-SA'...
remote: Enumerating objects: 532, done.
remote: Counting objects: 100% (178/178), done.
remote: Compressing objects: 100% (158/158), done.
remote: Total 532 (delta 76), reused 58 (delta 16), pack-reused 354 (from 1)
Receiving objects: 100% (532/532), 41.67 MiB | 18.67 MiB/s, done.
Resolving deltas: 100% (269/269), done.


In [2]:
%pip install -q "pulp>=2.7,<3"   # PuLP 2.x trae CBC incluido

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pulp
from IPython.display import display


## 2. Carga de datos

In [3]:
ruta_base = "Trabajo-Bebidas-del-Maipo-SA/Datos_crudos/"
ruta_resultados = "Trabajo-Bebidas-del-Maipo-SA/Resultados/"



---
## Parte A1. Balance de inventario por sku y semana.


In [4]:
# =====================================================================
# 2.1 DATOS CRUDOS DEL HORIZONTE TÁCTICO
# =====================================================================
maestro      = pd.read_csv(ruta_base + "maestro_sku.csv")
inv_inicial  = pd.read_csv(ruta_base + "inventario_inicial.csv")
capacidad    = pd.read_csv(ruta_base + "capacidad_horizonte.csv")
setups       = pd.read_csv(ruta_base + "setups_cip.csv")
compromisos  = pd.read_csv(ruta_base + "compromisos_semana_focal.csv")

# =====================================================================
# 2.2 TABLAS DE TRASPASO DE LA ENTREGA 1 (Parte C)
# =====================================================================
tabla_c1 = pd.read_csv(ruta_resultados + "merma_por_linea.csv")            # C.1  ρ_ℓ
tabla_c2 = pd.read_csv(ruta_resultados + "sobrellenado_por_formato.csv")   # C.2  jarabe extra
tabla_c3 = pd.read_csv(ruta_resultados + "pronostico_sku_horizonte.csv")   # C.3  d̂_it
tabla_c4 = pd.read_csv(ruta_resultados + "sigma_error_sku_horizonte.csv")  # C.4  σ̂_i,h

# C.1: ρ_ℓ por línea
rho = dict(zip(tabla_c1["Línea"], tabla_c1["ρ (proporción)"].astype(float)))

# C.2: litros de jarabe adicionales por caja, por formato (se usa en A3)
jarabe_extra_L_caja = dict(zip(tabla_c2["Formato (L)"].astype(float),
                               tabla_c2["Jarabe adicional (L/caja)"].astype(float)))

# C.3 y C.4: matriz SKU × (h=1..h=13) a formato largo (sku, h, valor)
def matriz_a_largo(tabla, nombre_valor):
    largo = tabla.melt(id_vars=["Familia", "SKU"], var_name="h", value_name=nombre_valor)
    largo["h"] = largo["h"].str.replace("h=", "", regex=False).astype(int)
    largo[nombre_valor] = largo[nombre_valor].astype(float)
    return largo.rename(columns={"SKU": "sku", "Familia": "familia"})

pronostico = matriz_a_largo(tabla_c3, "d_hat")
sigma_err  = matriz_a_largo(tabla_c4, "sigma")

# =====================================================================
# 2.3 VALIDACIONES DE CONSISTENCIA ENTRE E1 Y LOS DATOS DE E2
# =====================================================================
SKUS   = maestro["sku"].tolist()
LINEAS = sorted(maestro["linea_envasado"].unique())
T      = sorted(capacidad["t"].astype(int).tolist())

assert set(rho) == set(LINEAS), f"C.1 no trae ρ para todas las líneas: {rho}"
assert all(0 <= r < 1 for r in rho.values()), "ρ_ℓ debe estar en [0, 1)."
assert set(jarabe_extra_L_caja) == set(maestro["formato_L"]), "C.2 no cubre todos los formatos."
for nombre, df in [("C.3", pronostico), ("C.4", sigma_err)]:
    assert set(df["sku"]) == set(SKUS), f"{nombre}: los SKU no coinciden con maestro_sku."
    assert set(df["h"]) == set(T), f"{nombre}: los horizontes no son {T[0]}..{T[-1]}."
    assert not df.duplicated(["sku", "h"]).any(), f"{nombre}: hay (sku, h) duplicados."
assert (pronostico["d_hat"] >= 0).all(), "Hay pronósticos negativos en C.3."
assert (sigma_err["sigma"] >= 0).all(), "Hay σ̂ negativos en C.4."
assert set(inv_inicial["sku"]) == set(SKUS), "inventario_inicial no cubre los 9 SKU."

# El horizonte h del pronóstico corresponde a la semana t = h del plan
# (h = 1 es el lunes 28/09/2026, primera fila de capacidad_horizonte.csv).
d_hat = {(r.sku, r.h): r.d_hat for r in pronostico.itertuples()}

# Aviso: pronósticos que superan 3 veces la mediana del SKU en el horizonte
mediana_sku = pronostico.groupby("sku")["d_hat"].transform("median")
picos = pronostico[pronostico["d_hat"] > 3 * mediana_sku].merge(
    capacidad[["t", "fecha_lunes"]], left_on="h", right_on="t")
if len(picos):
    picos["veces_mediana"] = (picos["d_hat"]
                              / picos["sku"].map(pronostico.groupby("sku")["d_hat"].median())).round(1)
    print("⚠ Pronósticos atípicos (> 3 × mediana del SKU):")
    display(picos[["sku", "h", "fecha_lunes", "d_hat", "veces_mediana"]])

print(f"SKU: {len(SKUS)} | Líneas: {LINEAS} | Semanas: {T[0]}–{T[-1]} "
      f"({capacidad['fecha_lunes'].iloc[0]} a {capacidad['fecha_lunes'].iloc[-1]})")
print("ρ_ℓ:", rho)
print("Jarabe extra (L/caja):", jarabe_extra_L_caja)
print(f"Demanda pronosticada total del horizonte: {sum(d_hat.values()):,.0f} cajas")


⚠ Pronósticos atípicos (> 3 × mediana del SKU):


,sku,h,fecha_lunes,d_hat,veces_mediana
0,NEC-D15,11,2026-12-07,11654.0,5.8


SKU: 9 | Líneas: ['L1', 'L2'] | Semanas: 1–13 (2026-09-28 a 2026-12-21)
ρ_ℓ: {'L1': 0.019217, 'L2': 0.031146}
Jarabe extra (L/caja): {0.5: 0.049123, 1.5: 0.027606, 3.0: 0.034005}
Demanda pronosticada total del horizonte: 167,351 cajas


In [5]:
# =====================================================================
# PARTE A1 — BALANCE DE INVENTARIO POR SKU Y SEMANA (con backlogging y merma)
#   s[i,t-1] + (1-ρ_ℓ(i))·x[i,t] + b[i,t] = d[i,t] + s[i,t] + b[i,t-1]
# =====================================================================

# ---------------------------------------------------------------------
# 1. DATOS
# ---------------------------------------------------------------------
datos = maestro.set_index("sku").join(inv_inicial.set_index("sku"))

linea_de = datos["linea_envasado"].to_dict()                                # ℓ(i)
p_var    = datos["costo_variable_caja_clp"].astype(float).to_dict()         # p^i
h        = datos["costo_inventario_caja_semana_clp"].astype(float).to_dict()  # h^i
pi_atr   = datos["penalidad_atraso_caja_semana_clp"].astype(float).to_dict()  # π^i
s0       = datos["inventario_inicial_cajas"].astype(float).to_dict()        # s^i_0
b0       = {i: 0.0 for i in SKUS}                                           # b^i_0 (supuesto)
d        = d_hat                                                            # d^i_t
rend     = {i: 1 - rho[linea_de[i]] for i in SKUS}                          # 1 - ρ_ℓ(i)

tabla_a1_1 = pd.DataFrame({
    "SKU": SKUS,
    "Línea": [linea_de[i] for i in SKUS],
    "ρ_ℓ": [rho[linea_de[i]] for i in SKUS],
    "1 - ρ_ℓ": [rend[i] for i in SKUS],
    "s_0 (cajas)": [s0[i] for i in SKUS],
    "Σ d_t (cajas)": [sum(d[i, t] for t in T) for i in SKUS],
    "p (CLP/caja)": [p_var[i] for i in SKUS],
    "h (CLP/caja·sem)": [h[i] for i in SKUS],
    "π (CLP/caja·sem)": [pi_atr[i] for i in SKUS],
})
print("Tabla A1.1 — Datos del balance por SKU")
display(tabla_a1_1.round(6))

# ---------------------------------------------------------------------
# 2. MODELO, VARIABLES Y BALANCE
# ---------------------------------------------------------------------
# Las partes A2–A8 agregan restricciones a este mismo `modelo` y
# términos nuevos al diccionario `costos`.
modelo = pulp.LpProblem("BdM_LotSizing", pulp.LpMinimize)

x = pulp.LpVariable.dicts("x", (SKUS, T), lowBound=0)  # producción bruta
s = pulp.LpVariable.dicts("s", (SKUS, T), lowBound=0)  # inventario final
b = pulp.LpVariable.dicts("b", (SKUS, T), lowBound=0)  # backlog final

for i in SKUS:
    for t in T:
        s_prev = s0[i] if t == 1 else s[i][t - 1]
        b_prev = b0[i] if t == 1 else b[i][t - 1]
        modelo += (
            s_prev + rend[i] * x[i][t] + b[i][t] == d[i, t] + s[i][t] + b_prev,
            f"balance_{i}_{t}",
        )

costos = {
    "Producción (p·x)": pulp.lpSum(p_var[i] * x[i][t] for i in SKUS for t in T),
    "Inventario (h·s)": pulp.lpSum(h[i] * s[i][t] for i in SKUS for t in T),
    "Atraso (π·b)":     pulp.lpSum(pi_atr[i] * b[i][t] for i in SKUS for t in T),
}
modelo += pulp.lpSum(costos.values())

print(f"\nRestricciones de balance: {len(modelo.constraints)} "
      f"(= {len(SKUS)} SKU × {len(T)} semanas)")
print(f"Variables: {len(modelo.variables())} (x, s, b)")

# ---------------------------------------------------------------------
# 3. RESOLUCIÓN DE PRUEBA (solo A1, sin capacidad)
# ---------------------------------------------------------------------
# Sin capacidad, el óptimo se puede calcular a mano y sirve para
# comprobar el balance. Cada caja demandada en t se atiende de la forma
# más barata:
#   - producirla en t:            p / (1-ρ)
#   - atrasarla hasta el final:   π · (13 - t + 1)
# Si atrasar es más barato, el modelo no la produce. Esto es el efecto
# de fin de horizonte que corrige A7. El resto se produce lot-for-lot.
modelo.solve(pulp.PULP_CBC_CMD(msg=False))
print("\nEstado:", pulp.LpStatus[modelo.status])
print(f"Costo total: ${pulp.value(modelo.objective):,.0f}")
for nombre, termino in costos.items():
    print(f"  {nombre:<18} ${pulp.value(termino):>15,.0f}")

conviene_producir = {(i, t): pi_atr[i] * (T[-1] - t + 1) > p_var[i] / rend[i]
                     for i in SKUS for t in T}

filas = []
for i in SKUS:
    disponible = s0[i]
    for t in T:
        usa = min(disponible, d[i, t])
        disponible -= usa
        filas.append({
            "sku": i, "linea": linea_de[i], "t": t,
            "d": d[i, t],
            "x": x[i][t].varValue,
            "x_vendible": rend[i] * x[i][t].varValue,
            "merma": (1 - rend[i]) * x[i][t].varValue,
            "s": s[i][t].varValue,
            "b": b[i][t].varValue,
            "x_esperado": (d[i, t] - usa) / rend[i] if conviene_producir[i, t] else 0.0,
        })
balance_a1 = pd.DataFrame(filas)

# Comprobaciones
assert np.allclose(balance_a1["x"], balance_a1["x_esperado"], atol=1e-2), \
    "La producción no coincide con el óptimo calculado a mano."
assert not ((balance_a1["s"] > 1e-6) & (balance_a1["b"] > 1e-6)).any(), \
    "Hay inventario y atraso positivos a la vez."
print("Comprobaciones superadas: x coincide con el óptimo calculado a mano "
      "y nunca s > 0 y b > 0 a la vez.")

# Efecto de fin de horizonte
fin_horizonte = pd.DataFrame({
    "SKU": SKUS,
    "p/(1-ρ) (CLP)": [p_var[i] / rend[i] for i in SKUS],
    "semanas de atraso que cuestan menos": [int(p_var[i] / rend[i] // pi_atr[i]) for i in SKUS],
    "deja de producir desde t =": [min([t for t in T if not conviene_producir[i, t]], default="—")
                                   for i in SKUS],
    "b_13 (cajas)": [b[i][T[-1]].varValue for i in SKUS],
})
print("\nTabla A1.2 — Efecto de fin de horizonte sin restricción terminal")
display(fin_horizonte.round(1))

# ---------------------------------------------------------------------
# 4. RESULTADOS
# ---------------------------------------------------------------------
resumen = balance_a1.groupby(["linea", "sku"]).agg(
    demanda=("d", "sum"), produccion_bruta=("x", "sum"),
    produccion_vendible=("x_vendible", "sum"), merma=("merma", "sum"))
resumen.insert(1, "s_0", [s0[i] for i in resumen.index.get_level_values("sku")])
resumen["costo_merma_CLP"] = [m * p_var[i] for (l, i), m in resumen["merma"].items()]
print("\nTabla A1.3 — Producción y merma del horizonte por SKU, sin capacidad (cajas)")
display(resumen.round(1))

por_linea = resumen.groupby(level="linea").sum()
por_linea["merma_%"] = 100 * por_linea["merma"] / por_linea["produccion_bruta"]
print("Tabla A1.4 — Efecto de la merma por línea")
display(por_linea.round(1))

def por_semana(columna_valor):
    tabla = balance_a1.pivot(index=["linea", "sku"], columns="t", values=columna_valor)
    tabla.columns = [f"t={t}" for t in tabla.columns]
    return tabla.round(1)

print("Tabla A1.5 — Demanda d_t (cajas)")
display(por_semana("d"))
print("Tabla A1.6 — Producción bruta x_t (cajas)")
display(por_semana("x"))
print("Tabla A1.7 — Inventario final s_t (cajas)")
display(por_semana("s"))
print("Tabla A1.8 — Backlog final b_t (cajas)")
display(por_semana("b"))

Tabla A1.1 — Datos del balance por SKU


,SKU,Línea,ρ_ℓ,1 - ρ_ℓ,s_0 (cajas),Σ d_t (cajas),p (CLP/caja),h (CLP/caja·sem),π (CLP/caja·sem)
0,NEC-D05,L1,0.019217,0.980783,1024.0,13875.0,3840.0,95.0,1400.0
1,NEC-D15,L2,0.031146,0.968854,3162.0,36281.0,5760.0,95.0,1400.0
2,NEC-D30,L2,0.031146,0.968854,339.0,6851.0,7680.0,95.0,1400.0
3,CIT-L05,L1,0.019217,0.980783,1217.0,30808.0,2820.0,95.0,1400.0
4,CIT-L15,L2,0.031146,0.968854,1282.0,23257.0,4230.0,95.0,1400.0
5,CIT-L30,L2,0.031146,0.968854,255.0,5625.0,5640.0,95.0,1400.0
6,FUN-B05,L1,0.019217,0.980783,2135.0,29659.0,4860.0,95.0,1400.0
7,FUN-B15,L2,0.031146,0.968854,1261.0,18486.0,7290.0,95.0,1400.0
8,FUN-B30,L2,0.031146,0.968854,155.0,2509.0,9720.0,95.0,1400.0



Restricciones de balance: 117 (= 9 SKU × 13 semanas)
Variables: 351 (x, s, b)

Estado: Optimal
Costo total: $688,800,757
  Producción (p·x)   $    515,188,687
  Inventario (h·s)   $        124,070
  Atraso (π·b)       $    173,488,000
Comprobaciones superadas: x coincide con el óptimo calculado a mano y nunca s > 0 y b > 0 a la vez.

Tabla A1.2 — Efecto de fin de horizonte sin restricción terminal


,SKU,p/(1-ρ) (CLP),semanas de atraso que cuestan menos,deja de producir desde t =,b_13 (cajas)
0,NEC-D05,3915.2,2,12,1846.0
1,NEC-D15,5945.2,4,10,18024.0
2,NEC-D30,7926.9,5,9,2635.0
3,CIT-L05,2875.3,2,12,4867.0
4,CIT-L15,4366.0,3,11,5367.0
5,CIT-L30,5821.3,4,10,1962.0
6,FUN-B05,4955.2,3,11,6936.0
7,FUN-B15,7524.4,5,9,7110.0
8,FUN-B30,10032.5,7,7,1351.0



Tabla A1.3 — Producción y merma del horizonte por SKU, sin capacidad (cajas)


demanda     s_0  produccion_bruta  produccion_vendible  merma  \
linea sku                                                                      
L1    CIT-L05  30808.0  1217.0           25208.4              24724.0  484.4   
      FUN-B05  29659.0  2135.0           20991.4              20588.0  403.4   
      NEC-D05  13875.0  1024.0           11220.6              11005.0  215.6   
L2    CIT-L15  23257.0  1282.0           17141.9              16608.0  533.9   
      CIT-L30   5625.0   255.0            3517.6               3408.0  109.6   
      FUN-B15  18486.0  1261.0           10440.2              10115.0  325.2   
      FUN-B30   2509.0   155.0            1035.2               1003.0   32.2   
      NEC-D15  36281.0  3162.0           15580.3              15095.0  485.3   
      NEC-D30   6851.0   339.0            4001.6               3877.0  124.6   

               costo_merma_CLP  
linea sku                       
L1    CIT-L05        1366093.7  
      FUN-B05        1960483.0  
      NEC-D05         828006.9  
L2    CIT-L15        2258404.0  
      CIT-L30         617906.3  
      FUN-B15        2370485.8  
      FUN-B30         313408.8  
      NEC-D15        2795114.1  
      NEC-D30         957196.2

... (6 outputs left out)

Tabla A1.7 — Inventario final s_t (cajas)


t=1  t=2  t=3  t=4  t=5  t=6  t=7  t=8  t=9  t=10  t=11  \
linea sku                                                                   
L1    CIT-L05     0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0   0.0   0.0   
      FUN-B05     0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0   0.0   0.0   
      NEC-D05     0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0   0.0   0.0   
L2    CIT-L15     0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0   0.0   0.0   
      CIT-L30     0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0   0.0   0.0   
      FUN-B15     0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0   0.0   0.0   
      FUN-B30     0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0   0.0   0.0   
      NEC-D15  1306.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0   0.0   0.0   
      NEC-D30     0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0   0.0   0.0   

               t=12  t=13  
linea sku                  
L1    CIT-L05   0.0   0.0  
      FUN-B05   0.0   0.0  
      NEC-D05   0.0   0.0  
L2    CIT-L15   0.0   0.0  
      CIT-L30   0.0   0.0  
      FUN-B15   0.0   0.0  
      FUN-B30   0.0   0.0  
      NEC-D15   0.0   0.0  
      NEC-D30   0.0   0.0

Tabla A1.8 — Backlog final b_t (cajas)


t=1  t=2  t=3  t=4  t=5  t=6    t=7    t=8     t=9    t=10  \
linea sku                                                                   
L1    CIT-L05  0.0  0.0  0.0  0.0  0.0  0.0    0.0    0.0     0.0     0.0   
      FUN-B05  0.0  0.0  0.0  0.0  0.0  0.0    0.0    0.0     0.0     0.0   
      NEC-D05  0.0  0.0  0.0  0.0  0.0  0.0    0.0    0.0     0.0     0.0   
L2    CIT-L15  0.0  0.0  0.0  0.0  0.0  0.0    0.0    0.0     0.0     0.0   
      CIT-L30  0.0  0.0  0.0  0.0  0.0  0.0    0.0    0.0     0.0   477.0   
      FUN-B15  0.0  0.0  0.0  0.0  0.0  0.0    0.0    0.0  1422.0  2844.0   
      FUN-B30  0.0  0.0  0.0  0.0  0.0  0.0  193.0  386.0   579.0   772.0   
      NEC-D15  0.0  0.0  0.0  0.0  0.0  0.0    0.0    0.0     0.0  2025.0   
      NEC-D30  0.0  0.0  0.0  0.0  0.0  0.0    0.0    0.0   527.0  1054.0   

                  t=11     t=12     t=13  
linea sku                                 
L1    CIT-L05      0.0   2382.0   4867.0  
      FUN-B05   2310.0   4622.0   6936.0  
      NEC-D05      0.0    930.0   1846.0  
L2    CIT-L15   1789.0   3578.0   5367.0  
      CIT-L30    964.0   1459.0   1962.0  
      FUN-B15   4266.0   5688.0   7110.0  
      FUN-B30    965.0   1158.0   1351.0  
      NEC-D15  13679.0  16064.0  18024.0  
      NEC-D30   1581.0   2108.0   2635.0